# Generative morphology (AE + flow) on Euclid VIS — the 64 px tier

What the learned tier does on real data, one step at a time:

1. **The data** — a Euclid VIS quadrant (science, noise, flags, background).
2. **The selected galaxies** — the sources kept by the selection cuts, their
   flux compared with the flux distribution of the generative model, and
   generated galaxies re-convolved with the residual PSF.
3. **Galaxy-by-galaxy inference** — one 64x64 stamp per galaxy, latent code
   and recentring only: **no flux parameter, no shear**.
4. **The full library, selected galaxies only** — `MultiExposureScene` on the
   three full frames; everything else in the frame is left unmodelled.
5. **The full library, every tier** — 64 px sources through the generative
   model, 128/256 px sources through the parametric (Exponential) model.

Library conventions this notebook relies on (see `shine/euclid/scene.py` and
`shine/morphology/render.py`):

- `decode(z)` and every PSF stamp are **detector-grid arrays**. They are mapped
  onto the sky through the local WCS Jacobian, so the +57.9 deg rotation of the
  quadrant no longer rotates the model galaxies; the shear is applied on the sky.
- The learned tier draws with `method="no_pixel"` (the AE's training
  convention); the parametric tiers do too, because the PSF grid is sampled at
  the native pixel scale and so already contains the pixel response
  (`EuclidInferenceConfig.psf_includes_pixel`).
- Each source is drawn at its exact catalog position inside its stamp
  (`stamp_placement`), `dx`/`dy` being free corrections on top.

> **Why the *residual* PSF?** The AE was trained with `decode(z)` convolved by
> `psf_residual`, the kernel relating the true local PSF to a fixed isotropic
> reference PSF. `decode(z)` therefore already carries that reference PSF, and
> convolving it with the full local PSF would apply it twice. See
> `shine/morphology/psf_residual.py`.

## 0. Setup (Colab, GPU T4)

Run on a GPU runtime (*Runtime → Change runtime type → T4 GPU*): sections 3-5
re-render every galaxy through the decoder and JAX-GalSim FFTs at every
optimisation step. The setup cell does **not** pin `jax`, so Colab's CUDA build
is kept — check that `jax.devices()` reports a `cuda` device below.

The Euclid data (`data/EUC_VIS_SWL/`) and the AE/flow checkpoints
(`wandb_weights/`) come through **git-lfs**: without `git lfs pull` they are
130-byte pointer stubs.

In [ ]:
# Dependencies. Every version here is pinned on purpose; letting pip resolve
# the whole stack freely can hit "resolution-too-deep" and never finish, and
# two of the pins are load-bearing:
#   * equinox 0.13.6 -- the version the checkpoints were serialised with.
#   * flowjax >= 18 -- the MAF/RQS flow checkpoint (9i28jqsm) was trained with
#     it. flowjax 18.0.0 changed RationalQuadraticSpline's parameterisation
#     (40 parameters per latent dimension at knots=12, against 38 in 17.x), so
#     loading that checkpoint under flowjax 17.x fails outright with a shape
#     mismatch: "leaf ... has changed shape from (8, 608, 128) to (8, 640, 128)".
# NB: do NOT pin paramax==0.0.4 -- flowjax requires paramax>=0.0.5 and the pin
# makes the resolver fail outright.
!pip install -q "equinox==0.13.6" "einops>=0.8,<0.9"
!pip install -q "flowjax==18.0.0" "paramax>=0.0.5"
# Pinned rather than installed from git HEAD: an unpinned HEAD is what makes
# this notebook silently stop reproducing. If drawImage raises
# GalSimIncompatibleValuesError ("shape of array is inconsistent with provided
# bounds"), the jax / JAX-GalSim pair is mismatched -- try another release here.
!pip install -q "jax-galsim==2026.2.0"

In [ ]:
# Clone SHINE with its LFS payload (data + AE/flow checkpoints), then install.
# git-lfs is NOT preinstalled on Colab -- without the apt-get line, `git lfs`
# fails with "'lfs' is not a git command" and the clone silently brings down
# 130-byte pointer stubs instead of the FITS files and the .eqx checkpoints.
!apt-get -qq install -y git-lfs
!git lfs install

# Absolute path, and re-runnable: a plain `git clone ... SHINE` followed by
# `%cd SHINE` nests a second checkout at /content/SHINE/SHINE the second time
# the cell is run, and then fails outright on the third.
!test -d /content/SHINE/.git || git clone -b GenGal64 https://github.com/VincentB03/SHINE.git /content/SHINE
%cd /content/SHINE
!git lfs pull
!pip install -q -e .

# Fail loudly here rather than three cells later: a pointer stub is a few
# hundred bytes, the real checkpoint is ~4 MB.
import pathlib
_ckpt = pathlib.Path("wandb_weights/9i28jqsm/epoch_500/model_checkpoint_500.eqx")
assert _ckpt.exists(), f"{_ckpt} missing -- the clone did not bring it down"
assert _ckpt.stat().st_size > 1_000_000, (
    f"{_ckpt} is {_ckpt.stat().st_size} bytes -- git-lfs did not fetch it")
print("git-lfs payload OK")

In [ ]:
import logging
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message=".*complex128.*", module="jax_galsim")

import jax
import jax.numpy as jnp
import jax_galsim as galsim
import matplotlib.pyplot as plt
import numpy as np
import numpyro
import numpyro.distributions as dist
import paramax
from astropy.io import fits
from numpyro.handlers import condition

from shine.config import InferenceConfig, MAPConfig
from shine.euclid.config import EuclidDataConfig, EuclidInferenceConfig, SourceSelectionConfig
from shine.euclid.data_loader import EuclidDataLoader
from shine.euclid.plots import plot_exposure_comparison
from shine.euclid.scene import MultiExposureScene, render_model_images, stamp_placement
from shine.inference import Inference
from shine.morphology.config import LearnedMorphologyConfig
from shine.morphology.loader import load_frozen_autoencoder, load_frozen_flow
from shine.morphology.prior import sample_latent_codes
from shine.morphology.render import render_decoded_galaxy

%matplotlib inline

# Run from either the repo root or notebooks/.
REPO_ROOT = Path.cwd() if (Path.cwd() / "shine").is_dir() else Path.cwd().parent
DATA_DIR = REPO_ROOT / "data" / "EUC_VIS_SWL"
QUADRANT = "3-4.F"
PIXEL_SCALE = 0.1
STAMP = 64                      # the learned tier

# Frozen AE + flow checkpoints (committed in the repo, git-lfs).
AE_CHECKPOINT_DIR = str(REPO_ROOT / "wandb_weights" / "i344nq38" / "epoch_2000")
AE_EPOCH = 2000
FLOW_CHECKPOINT_DIR = str(REPO_ROOT / "wandb_weights" / "9i28jqsm" / "epoch_500")
FLOW_EPOCH = 500

# Section 2 selection: a faint SNR band drawn at random -- the part of the
# population the flow was trained on (see section 2).
MIN_SNR, MAX_SNR = 12.0, 25.0
MAX_SOURCES = 60
SELECTION_ORDER, SELECTION_SEED = "random", 0

# Section 5 selection: the standard cuts, every tier.
ALL_MIN_SNR = 10.0
ALL_MAX_SOURCES = 400           # raise it if GPU memory allows
ALL_SELECTION_ORDER = "brightest"  # the sources that dominate the residual image

# MAP settings (sections 3-5). z_base is a standard-normal site: the learning
# rate bounds how far it can move (~ FIT_LR * FIT_STEPS per dimension).
FIT_STEPS = 400
FIT_LR = 0.01
RNG_SEED = 42

N_PRIOR = 2000   # draws from the generative model for the flux distribution
N_SHOW = 8       # generated galaxies displayed

# Flag bits not modelled by the scene (INVALID, GHOST, STARSIGNAL,
# SATURATEDSTAR): left out of the residual statistics.
RESIDUAL_EXCLUDE_BITS = 0x1 | (1 << 5) | (1 << 18) | (1 << 19)

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
logging.getLogger("jax").setLevel(logging.WARNING)

print("Repo root  :", REPO_ROOT)
print("JAX devices:", jax.devices())
if jax.devices()[0].platform != "gpu":
    print("WARNING: running on CPU -- sections 3-5 will be slow.")

for path in [DATA_DIR / "PSF_3-4-F_residual.fits.gz",
             Path(AE_CHECKPOINT_DIR) / f"model_checkpoint_{AE_EPOCH}.eqx"]:
    assert path.exists(), f"missing: {path}"
    assert path.stat().st_size > 10_000, f"{path} looks like a git-lfs pointer -- run `git lfs pull`"

## 1. The data

Real Euclid Q1 VIS data: observation 2704 (NGC 6505), quadrant **3-4.F**,
three dithered exposures of 560.52 s. Per exposure, three 2048x2066 planes:
`SCI` (ADU, **not** background-subtracted), `RMS` (per-pixel noise sigma, the
likelihood weights) and `FLG` (data-quality bitmask). Separate products: the
pipeline background maps (`BKG`), the PSF model as a 9x9 grid of 21x21 stamps
(`PSF_3-4-F.fits.gz`), the residual-PSF grid for the learned tier, and the
MER catalogue driving the selection.

What SHINE fits is `SCI - BKG` pixel by pixel (`EuclidExposure.prepare_image_data`),
with `sigma = 1e10` (zero weight) on pixels flagged `FLG & bad_pixel_mask`.

In [ ]:
exposure_paths = sorted(str(p) for p in DATA_DIR.glob("EUC_VIS_SWL-DET-*_3-4-F.fits.gz"))
bkg_paths = sorted(str(p) for p in DATA_DIR.glob("EUC_VIS_SWL-BKG-*_3-4-F.fits.gz"))
assert len(exposure_paths) == 3 and len(bkg_paths) == 3

with fits.open(exposure_paths[0]) as hdul:
    sci_raw = hdul[f"{QUADRANT}.SCI"].data.astype(np.float32)
    rms_raw = hdul[f"{QUADRANT}.RMS"].data.astype(np.float32)
    flg_raw = hdul[f"{QUADRANT}.FLG"].data.astype(np.int32)
    hdr = hdul[f"{QUADRANT}.SCI"].header
with fits.open(bkg_paths[0]) as hdul:
    bkg_raw = hdul[QUADRANT].data.astype(np.float32)

for key in ("EXPTIME", "GAIN", "MAGZEROP"):
    print(f"  {key:9s} = {hdr.get(key)}")
print(f"  shape     = {sci_raw.shape}, flagged = {(flg_raw != 0).mean() * 100:.2f}% of pixels")

panels = [
    (np.arcsinh(sci_raw), "SCI [arcsinh(ADU)]", "gray_r"),
    (np.arcsinh(sci_raw - bkg_raw), "SCI - BKG, what is fitted [arcsinh(ADU)]", "gray_r"),
    (bkg_raw, "Background map [ADU]", "viridis"),
    (rms_raw, "RMS noise [ADU]", "magma"),
    (flg_raw != 0, "Flagged pixels", "gray"),
]
fig, axes = plt.subplots(1, len(panels), figsize=(5.2 * len(panels), 5.6))
for ax, (img, title, cmap) in zip(axes, panels):
    arr = np.asarray(img, dtype=np.float32)
    vmin, vmax = np.percentile(arr[np.isfinite(arr)], [1, 99])
    im = ax.imshow(arr, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
    ax.set_title(title, fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle(f"Quadrant {QUADRANT}, dither 0 — {sci_raw.shape[1]}x{sci_raw.shape[0]} px @ 0.1\"/px",
             fontsize=13)
fig.tight_layout()
plt.show()

## 2. The selected galaxies

`EuclidDataLoader` applies the `SourceSelectionConfig` cuts to the MER
catalogue (SNR, VIS detection, spurious / point-source flags, detection
quality bitmask) and assigns each source the smallest stamp tier that holds
`2 * (3 * hlr_px + 10.5)` pixels. With `galaxy_stamp_sizes=[64]`, sources too
large for 64 px are dropped, so every kept galaxy is on the learned tier.

The cut is a **faint SNR band drawn at random**: the flow was trained on a
magnitude-limited population, so the brightest sources of the quadrant (the
old default, `selection_order="brightest"`) lie far outside its prior.

In [ ]:
learned_morphology = LearnedMorphologyConfig(
    enabled=True,
    ae_checkpoint_dir=AE_CHECKPOINT_DIR,
    ae_epoch=AE_EPOCH,
    flow_checkpoint_dir=FLOW_CHECKPOINT_DIR,
    flow_epoch=FLOW_EPOCH,
    apply_to_stamp_size=STAMP,
    psf_residual_path=str(DATA_DIR / "PSF_3-4-F_residual.fits.gz"),
)
data_config = EuclidDataConfig(
    exposure_paths=exposure_paths,
    psf_path=str(DATA_DIR / "PSF_3-4-F.fits.gz"),
    catalog_path=str(DATA_DIR / "catalogue_3-4-F.fits.gz"),
    background_paths=bkg_paths,
    quadrant=QUADRANT,
    pixel_scale=PIXEL_SCALE,
)
inference_config = InferenceConfig(
    method="map",
    map_config=MAPConfig(enabled=True, num_steps=FIT_STEPS, learning_rate=FIT_LR),
    rng_seed=RNG_SEED,
)
config = EuclidInferenceConfig(
    data=data_config,
    sources=SourceSelectionConfig(
        min_snr=MIN_SNR, max_snr=MAX_SNR, max_sources=MAX_SOURCES,
        selection_order=SELECTION_ORDER, selection_seed=SELECTION_SEED,
    ),
    inference=inference_config,
    galaxy_stamp_sizes=[STAMP],
    learned_morphology=learned_morphology,
)

data = EuclidDataLoader(config).load()
assert int(np.asarray(data.source_stamp_tier).max()) == 0

# Frozen generative model: AE decoder + normalizing-flow prior on its latents.
ae = load_frozen_autoencoder(AE_CHECKPOINT_DIR, AE_EPOCH)
flow = load_frozen_flow(FLOW_CHECKPOINT_DIR, FLOW_EPOCH)
LATENT_FLAT = int(np.prod(flow.latent_dim))
assert ae.nx == ae.ny == STAMP and abs(ae.scale - PIXEL_SCALE) < 1e-6

# What sample_latent_codes does to its standard-normal z_base site: rescale to
# the flow's own base distribution, then push through the flow.
_base = paramax.unwrap(flow.flow).base_dist
BASE_LOC, BASE_SCALE = jnp.asarray(_base.loc), jnp.asarray(_base.scale)


def latent_from_base(z_base):
    return flow.unflatten_latent(jax.vmap(flow.forward)(BASE_LOC + BASE_SCALE * z_base))


decode_batch = jax.jit(jax.vmap(lambda z: ae.decode(z, key=None)[0]))
GSP = galsim.GSParams(minimum_fft_size=2 * STAMP, maximum_fft_size=2 * STAMP)  # as the library


def render_stamps(gal, psf, wcs, off_x, off_y, g1=0.0, g2=0.0):
    """The library's learned-tier renderer, vmapped over stamps."""
    def one(g_i, p_i, w_i, ox, oy):
        return render_decoded_galaxy(g_i, g1, g2, p_i, w_i, ox, oy, True,
                                     STAMP, PIXEL_SCALE, GSP)
    return jax.vmap(one)(gal, psf, wcs, off_x, off_y)


print(f"\nSources kept : {data.n_sources} (all on the {STAMP} px learned tier)")
print(f"Latent space : {list(flow.latent_dim)} ({LATENT_FLAT} values)")

In [ ]:
# The galaxies visible in exposure 0, and their 64x64 cutouts: exactly the
# stamps the scene model pastes them into (stamp_placement), so the cutout
# and the model share one pixel grid.
image0 = np.asarray(data.images[0])
sigma0 = np.asarray(data.noise_sigma[0])
fit_idx = np.where(np.asarray(data.source_visible[:, 0]))[0]
pos_fit = np.asarray(data.pixel_positions[fit_idx, 0])
cx, cy, sub_x, sub_y = (np.asarray(a) for a in
                        stamp_placement(pos_fit, STAMP, data.image_nx, data.image_ny))
obs_stamps = jnp.asarray(np.stack([image0[y:y + STAMP, x:x + STAMP] for x, y in zip(cx, cy)]))
sigma_stamps = jnp.asarray(np.stack([sigma0[y:y + STAMP, x:x + STAMP] for x, y in zip(cx, cy)]))
psf_res_fit = jnp.asarray(data.psf_residual_images[fit_idx, 0])
wcs_fit = jnp.asarray(data.wcs_jacobians[fit_idx, 0])
n_fit = len(fit_idx)
print(f"{n_fit} / {data.n_sources} selected galaxies are visible in exposure 0")

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(np.arcsinh(image0), origin="lower", cmap="gray_r",
          vmin=np.percentile(np.arcsinh(image0), 1), vmax=np.percentile(np.arcsinh(image0), 99.5))
ax.scatter(pos_fit[:, 0], pos_fit[:, 1], s=80, facecolors="none", edgecolors="#E53935", lw=1.3)
ax.set_title(f"Selected galaxies in exposure 0 (N={n_fit})")
plt.show()

n_cols = 8
n_rows = int(np.ceil(n_fit / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.1 * n_cols, 2.3 * n_rows), squeeze=False)
for k, ax in enumerate(axes.ravel()):
    ax.axis("off")
    if k < n_fit:
        ax.imshow(np.arcsinh(np.asarray(obs_stamps[k])), origin="lower", cmap="gray_r")
        ax.set_title(f"#{fit_idx[k]} | {float(data.catalog_flux_adu[fit_idx[k]]):.0f} ADU", fontsize=8)
fig.suptitle("Observed 64x64 stamps (exposure 0, arcsinh)", fontsize=12)
fig.tight_layout()
plt.show()

### Flux: selected galaxies against the generative model

The flux of a generated galaxy is the sum of `decode(z)`, with `z` drawn from
the flow prior exactly as the inference draws it (`z_base ~ N(0, 1)` through
`latent_from_base`). The PSF convolution preserves it (unit-sum kernels). There
is **no separate flux parameter** on the learned tier: the latent code carries
the amplitude, so the prior must cover the observed fluxes by itself.

In [ ]:
keys = jax.random.split(jax.random.key(0), N_PRIOR // 250 + 1)
gen_flux = np.concatenate([
    np.asarray(decode_batch(latent_from_base(jax.random.normal(k, (250, LATENT_FLAT)))).sum(axis=(1, 2)))
    for k in keys
])[:N_PRIOR]
sel_flux = np.asarray(data.catalog_flux_adu)

bins = np.linspace(np.log10(min(gen_flux.min(), sel_flux.min())) - 0.1,
                   np.log10(max(gen_flux.max(), sel_flux.max())) + 0.1, 40)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(np.log10(gen_flux), bins=bins, density=True, color="#8E24AA", alpha=0.55,
        edgecolor="k", label=f"generative model, {N_PRIOR} draws")
ax.hist(np.log10(sel_flux), bins=bins, density=True, color="#FB8C00", alpha=0.6,
        edgecolor="k", label=f"selected galaxies, catalogue ({data.n_sources})")
ax.set_xlabel("log10(total flux [ADU])"); ax.set_ylabel("density"); ax.legend()
ax.set_title("Flux: prior population vs. the selected galaxies")
plt.show()

for label, f in (("generative model", gen_flux), ("selected galaxies", sel_flux)):
    lo, med, hi = np.percentile(f, [5, 50, 95])
    print(f"{label:18s}: median {med:8.0f} ADU, 5-95th [{lo:.0f}, {hi:.0f}]")
q = np.mean((sel_flux[:, None] > gen_flux[None, :]), axis=1)
print(f"prior quantile of each selected galaxy's flux: median {np.median(q):.2f}, "
      f"min {q.min():.2f}, max {q.max():.2f}  (0 or 1 = outside the prior)")

### Generated galaxies, re-convolved with the residual PSF

Eight draws from the generative model, each rendered with the library's
learned-tier renderer (`render_decoded_galaxy`) through the residual PSF and
the WCS Jacobian at the position of one of the selected galaxies — i.e. what
the scene would put in that stamp for that latent code. The observed stamp at
the same position is shown for comparison.

In [ ]:
n_show = min(N_SHOW, n_fit)
z_show = latent_from_base(jax.random.normal(jax.random.key(1), (n_show, LATENT_FLAT)))
gen_show = decode_batch(z_show)
conv_show = render_stamps(gen_show, psf_res_fit[:n_show], wcs_fit[:n_show],
                          jnp.zeros(n_show), jnp.zeros(n_show))

rows = [
    (gen_show, "decode(z)", "inferno", True),
    (psf_res_fit[:n_show], "residual PSF", "viridis", False),
    (conv_show, "decode(z) * PSF_res", "inferno", True),
    (obs_stamps[:n_show], "observed stamp", "gray_r", True),
]
fig, axes = plt.subplots(len(rows), n_show, figsize=(2.1 * n_show, 2.2 * len(rows)), squeeze=False)
for r, (arr, label, cmap, stretch) in enumerate(rows):
    for k in range(n_show):
        a = np.asarray(arr[k])
        axes[r, k].imshow(np.arcsinh(a) if stretch else a, origin="lower", cmap=cmap)
        axes[r, k].set_xticks([]); axes[r, k].set_yticks([])
    axes[r, 0].set_ylabel(label, fontsize=9)
for k in range(n_show):
    axes[0, k].set_title(f"flux {float(gen_show[k].sum()):.0f}", fontsize=8)
fig.suptitle("Generated galaxies re-convolved with the residual PSF (arcsinh)", fontsize=12)
fig.tight_layout()
plt.show()

ratio = np.asarray(conv_show.sum(axis=(1, 2)) / gen_show.sum(axis=(1, 2)))
print("flux after / before convolution:", np.round(ratio, 3))

## 3. Galaxy-by-galaxy inference — no flux parameter, no shear

Each galaxy visible in exposure 0 is fitted **on its own 64x64 stamp**, with
the library renderer. Free per galaxy: the latent code `z` (flow prior) and a
recentring `dx`, `dy` (`N(0, 0.05")`, the library's prior) on top of the
catalog position. **Nothing else**: no flux parameter — the amplitude has to
come out of `z` — and the shear is held at zero, so nothing about the fit can
leak into a shear estimate.

The question is purely: *can the generative model reproduce these galaxies?*
Read the chi²/pixel (1 = noise-limited) and whether the fitted flux follows the
catalogue flux. Neighbours inside a stamp are not modelled (nor masked), so the
worst fits are often contaminated stamps rather than decoder failures — look at
the panels.

In [ ]:
def stamp_model(observed_data=None):
    with numpyro.plate("galaxies", n_fit):
        z = sample_latent_codes("z", flow, n_fit)
        dx = numpyro.sample("dx", dist.Normal(0.0, 0.05))   # arcsec
        dy = numpyro.sample("dy", dist.Normal(0.0, 0.05))
    stamps = render_stamps(decode_batch(z), psf_res_fit, wcs_fit,
                           sub_x + dx / PIXEL_SCALE, sub_y + dy / PIXEL_SCALE)
    numpyro.sample("obs", dist.Normal(stamps, sigma_stamps).to_event(3), obs=observed_data)


t0 = time.time()
est3 = Inference(stamp_model, inference_config).run_map(
    jax.random.PRNGKey(RNG_SEED), observed_data=obs_stamps,
    map_config=inference_config.map_config,
    init_params={"z_base": jnp.zeros((n_fit, LATENT_FLAT)),
                 "dx": jnp.zeros(n_fit), "dy": jnp.zeros(n_fit)},
)
print(f"{n_fit} stamps, {FIT_STEPS} steps: {time.time() - t0:.0f} s")

dec3 = decode_batch(latent_from_base(est3["z_base"]))
model3 = render_stamps(dec3, psf_res_fit, wcs_fit,
                       sub_x + est3["dx"] / PIXEL_SCALE, sub_y + est3["dy"] / PIXEL_SCALE)
valid = np.asarray(sigma_stamps) < 1e9
chi3 = np.asarray((obs_stamps - model3) / sigma_stamps)
chi2_3 = np.array([np.mean(chi3[k][valid[k]] ** 2) for k in range(n_fit)])
flux3 = np.asarray(dec3.sum(axis=(1, 2)))
print(f"chi2/pixel: median {np.median(chi2_3):.2f}, 16-84th "
      f"[{np.percentile(chi2_3, 16):.2f}, {np.percentile(chi2_3, 84):.2f}]")
print(f"recentring |d| [px]: median "
      f"{np.median(np.hypot(est3['dx'], est3['dy'])) / PIXEL_SCALE:.2f}")

In [ ]:
cat_flux = np.asarray(data.catalog_flux_adu[fit_idx])
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(np.log10(chi2_3), bins=15, color="#1E88E5", edgecolor="k")
axes[0].axvline(0, color="k", ls="--", label="chi2/px = 1")
axes[0].set_xlabel("log10(chi2 / pixel)"); axes[0].set_title("Fit quality per galaxy"); axes[0].legend()
axes[1].loglog(cat_flux, flux3, "o", color="#43A047")
lim = [min(cat_flux.min(), flux3.min()) * 0.7, max(cat_flux.max(), flux3.max()) * 1.4]
axes[1].plot(lim, lim, "k--", label="1:1")
axes[1].set_xlabel("catalogue flux [ADU]"); axes[1].set_ylabel("fitted model flux [ADU]")
axes[1].set_title("Amplitude carried by z alone"); axes[1].legend()
fig.tight_layout()
plt.show()
print(f"fitted / catalogue flux: median {np.median(flux3 / cat_flux):.2f}, "
      f"16-84th [{np.percentile(flux3 / cat_flux, 16):.2f}, {np.percentile(flux3 / cat_flux, 84):.2f}]")

order = np.argsort(chi2_3)
n_panel = min(12, n_fit)
show = order[np.linspace(0, n_fit - 1, n_panel).round().astype(int)]   # best to worst
fig, axes = plt.subplots(4, n_panel, figsize=(1.9 * n_panel, 8.2), squeeze=False)
for c, k in enumerate(show):
    lim = float(np.percentile(np.abs(chi3[k]), 99))
    panels = [(np.arcsinh(np.asarray(obs_stamps[k])), "gray_r", None),
              (np.arcsinh(np.asarray(model3[k])), "gray_r", None),
              (chi3[k], "RdBu_r", lim),
              (np.arcsinh(np.asarray(dec3[k])), "inferno", None)]
    for r, (a, cmap, l) in enumerate(panels):
        axes[r, c].imshow(a, origin="lower", cmap=cmap,
                          vmin=-l if l else None, vmax=l if l else None)
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
    axes[0, c].set_title(f"#{fit_idx[k]}\nchi2/px {chi2_3[k]:.1f}", fontsize=8)
for r, label in enumerate(["observed", "MAP model", "chi", "decode(z_MAP)"]):
    axes[r, 0].set_ylabel(label, fontsize=9)
fig.suptitle("Galaxy-by-galaxy fits, best (left) to worst (right)", fontsize=12)
fig.tight_layout()
plt.show()

## 4. The full library — selected galaxies only

The same galaxies through the code a SHINE run executes: `MultiExposureScene`
builds the NumPyro model on the **three full frames**, each galaxy's stamp is
rendered per exposure (its own PSF, WCS and position) and pasted into a
full-size model image, and the likelihood is evaluated on every pixel. The
latent code is shared by a galaxy's exposures.

Everything that is not a selected galaxy — brighter galaxies, stars, fainter
sources — is **not modelled**: it stays in the residual. The shear is held at
zero (`numpyro.handlers.condition`), as in section 3; drop the `condition` to
let the model infer it.

In [ ]:
def learned_indices(cfg, dat):
    tier = cfg.galaxy_stamp_sizes.index(cfg.learned_morphology.apply_to_stamp_size)
    return np.where(np.asarray(dat.source_stamp_tier) == tier)[0]


def fit_scene(cfg, dat):
    """MAP of the library scene model with the shear held at zero."""
    scene = MultiExposureScene(cfg, dat)
    model = condition(scene.build_model(), data={"g1": 0.0, "g2": 0.0})
    n, lidx = dat.n_sources, learned_indices(cfg, dat)
    init = {
        "flux": jnp.asarray(dat.catalog_flux_adu), "hlr": jnp.asarray(dat.catalog_hlr_arcsec),
        "e1": jnp.zeros(n), "e2": jnp.zeros(n), "dx": jnp.zeros(n), "dy": jnp.zeros(n),
        "z_base": jnp.zeros((len(lidx), LATENT_FLAT)),
    }
    t0 = time.time()
    est = Inference(model, cfg.inference).run_map(
        jax.random.PRNGKey(RNG_SEED), observed_data=dat.images,
        map_config=cfg.inference.map_config, init_params=init,
    )
    print(f"scene MAP: {n} sources ({len(lidx)} learned), {dat.n_exposures} exposures, "
          f"{FIT_STEPS} steps: {time.time() - t0:.0f} s")

    params = {k: v for k, v in est.items() if k != "z_base"}
    params["g1"], params["g2"] = 0.0, 0.0
    z = jnp.zeros((n, *flow.latent_dim))
    if len(lidx):
        z = z.at[lidx].set(latent_from_base(est["z_base"]))
    params["z"] = z
    images = render_model_images(
        params, dat, pixel_scale=PIXEL_SCALE, stamp_sizes=cfg.galaxy_stamp_sizes,
        ae=scene.ae, learned_tier_idx=cfg.galaxy_stamp_sizes.index(STAMP),
        psf_includes_pixel=cfg.psf_includes_pixel,
    )
    return est, np.asarray(images)


def frame_chi2(dat, model_images, j=0):
    keep = np.asarray((dat.flag_maps[j] & RESIDUAL_EXCLUDE_BITS) == 0)
    chi = (np.asarray(dat.images[j]) - model_images[j]) / np.asarray(dat.noise_sigma[j])
    return chi, float(np.mean(chi[keep] ** 2))


est4, model4 = fit_scene(config, data)
chi4, chi2_frame4 = frame_chi2(data, model4)
print(f"exposure 0: chi2/pixel over the whole frame = {chi2_frame4:.3f}")

In [ ]:
fig = plot_exposure_comparison(
    observed=data.images[0], model=model4[0], noise_sigma=data.noise_sigma[0],
    mask=data.masks[0], exposure_idx=0,
    residual_mask=(data.flag_maps[0] & RESIDUAL_EXCLUDE_BITS) == 0,
)
plt.show()

# The same stamps as section 3, cut out of the library's full-frame model.
lib_stamps = np.stack([model4[0, y:y + STAMP, x:x + STAMP] for x, y in zip(cx, cy)])
chi_lib = (np.asarray(obs_stamps) - lib_stamps) / np.asarray(sigma_stamps)
chi2_lib = np.array([np.mean(chi_lib[k][valid[k]] ** 2) for k in range(n_fit)])
print(f"chi2/pixel on the selected stamps: library {np.median(chi2_lib):.2f} "
      f"(median) vs. galaxy-by-galaxy {np.median(chi2_3):.2f}")

fig, axes = plt.subplots(4, n_panel, figsize=(1.9 * n_panel, 8.2), squeeze=False)
for c, k in enumerate(show):
    lim = float(np.percentile(np.abs(chi_lib[k]), 99))
    panels = [(np.arcsinh(np.asarray(obs_stamps[k])), "gray_r", None),
              (np.arcsinh(np.asarray(model3[k])), "gray_r", None),
              (np.arcsinh(lib_stamps[k]), "gray_r", None),
              (chi_lib[k], "RdBu_r", lim)]
    for r, (a, cmap, l) in enumerate(panels):
        axes[r, c].imshow(a, origin="lower", cmap=cmap,
                          vmin=-l if l else None, vmax=l if l else None)
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
    axes[0, c].set_title(f"#{fit_idx[k]}\nlib chi2/px {chi2_lib[k]:.1f}", fontsize=8)
for r, label in enumerate(["observed", "section 3 model", "library model", "library chi"]):
    axes[r, 0].set_ylabel(label, fontsize=9)
fig.suptitle("Library fit on the full frames (exposure 0), same stamps as section 3", fontsize=12)
fig.tight_layout()
plt.show()

## 5. The full library — every tier

Now the whole scene: the standard selection cuts (no SNR ceiling, no size
limit beyond the largest stamp) and `galaxy_stamp_sizes=[64, 128, 256]`.
Sources on the **64 px tier go through the generative model** (latent code, no
flux parameter); sources on the **128 and 256 px tiers go through the
parametric model** (Exponential profile with flux, half-light radius, intrinsic
ellipticity, catalog-centred priors). `ALL_MAX_SOURCES` caps the number of
sources (the brightest by default, the ones that dominate the residual image).

Note the 64 px tier now also holds bright compact sources, well above the
flux range the generative prior covers (section 2) — expect those to fit worse
than the faint ones.

In [ ]:
config_all = EuclidInferenceConfig(
    data=data_config,
    sources=SourceSelectionConfig(
        min_snr=ALL_MIN_SNR, max_sources=ALL_MAX_SOURCES,
        selection_order=ALL_SELECTION_ORDER, selection_seed=SELECTION_SEED,
    ),
    inference=inference_config,
    galaxy_stamp_sizes=[64, 128, 256],
    learned_morphology=learned_morphology,
)
data_all = EuclidDataLoader(config_all).load()
tiers_all = np.asarray(data_all.source_stamp_tier)
for t, ss in enumerate(config_all.galaxy_stamp_sizes):
    kind = "generative" if ss == STAMP else "parametric"
    print(f"  {ss:3d} px tier: {(tiers_all == t).sum():4d} sources ({kind})")

est5, model5 = fit_scene(config_all, data_all)
chi5, chi2_frame5 = frame_chi2(data_all, model5)
print(f"exposure 0: chi2/pixel over the whole frame = {chi2_frame5:.3f} "
      f"(selected galaxies only, section 4: {chi2_frame4:.3f})")

In [ ]:
fig = plot_exposure_comparison(
    observed=data_all.images[0], model=model5[0], noise_sigma=data_all.noise_sigma[0],
    mask=data_all.masks[0], exposure_idx=0,
    residual_mask=(data_all.flag_maps[0] & RESIDUAL_EXCLUDE_BITS) == 0,
)
plt.show()

# chi2/pixel inside each tier's stamps (exposure 0).
chi5_valid = np.asarray(data_all.noise_sigma[0]) < 1e9
for t, ss in enumerate(config_all.galaxy_stamp_sizes):
    idx = np.where((tiers_all == t) & np.asarray(data_all.source_visible[:, 0]))[0]
    if len(idx) == 0:
        continue
    pos = np.asarray(data_all.pixel_positions[idx, 0])
    tx, ty, _, _ = (np.asarray(a) for a in stamp_placement(pos, ss, data_all.image_nx,
                                                          data_all.image_ny))
    per = [np.mean(chi5[y:y + ss, x:x + ss][chi5_valid[y:y + ss, x:x + ss]] ** 2)
           for x, y in zip(tx, ty)]
    kind = "generative" if ss == STAMP else "parametric"
    print(f"  {ss:3d} px tier ({kind:10s}): chi2/pixel in stamps, median {np.median(per):7.2f} "
          f"over {len(idx)} sources")

# Side by side on the same region: residual with the selected galaxies only
# (section 4) and with every tier (section 5).
yc, xc = data.image_ny // 2, data.image_nx // 2
half = 256
sl = (slice(yc - half, yc + half), slice(xc - half, xc + half))
fig, axes = plt.subplots(1, 3, figsize=(17, 5.8))
obs_region = np.asarray(data.images[0])[sl]
axes[0].imshow(np.arcsinh(obs_region), origin="lower", cmap="gray_r",
               vmin=np.percentile(np.arcsinh(obs_region), 1),
               vmax=np.percentile(np.arcsinh(obs_region), 99.5))
axes[0].set_title("observed (central 512x512)")
for ax, chi, title in ((axes[1], chi4, "chi, selected galaxies only (section 4)"),
                       (axes[2], chi5, "chi, every tier (section 5)")):
    ax.imshow(chi[sl], origin="lower", cmap="RdBu_r", vmin=-5, vmax=5)
    ax.set_title(title)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout()
plt.show()

## 6. Reading the results

- **Section 2** — the prior quantiles of the selected fluxes: if they sit
  inside (0, 1), the generative model can in principle produce galaxies that
  bright without a flux parameter.
- **Section 3** — the chi²/pixel and the fitted-vs-catalogue flux. A chi² near
  1 with a flux ratio near 1 means `z` alone reproduces the galaxies. A flux
  ratio that is systematically off, or widely scattered, is the case for a
  per-galaxy flux parameter on the learned tier (issue #6b); a chi² far above 1
  on isolated stamps points at the decoder itself (latent size, #2's FFT
  wrap-around baked into the weights).
- **Section 4** — the library on the full frames should land close to
  section 3 on the same stamps; a large gap would mean something differs
  between the stamp fit and the scene (placement, PSF, WCS, multi-exposure).
- **Section 5** — how much of the frame the model explains once every tier
  is on, and how the generative 64 px tier compares with the parametric tiers.
  The residual left over (stars, flagged regions, sources beyond
  `ALL_MAX_SOURCES`) is what is not modelled at all.

None of this infers a shear: that comes after the model reproduces the
galaxies, and after a validation on injected known shear.